In [0]:
# ============================================
# Inflate fact tables (~1000 ±100 rows) with type-safe casts for Delta
# - Serverless-safe (no RDD calls, no UDFs)
# - Detects target column types and casts generated values accordingly
# - Overwrites only: SalesAmount, TaxAmt, Freight, Discount, OrderDate, DueDate, ShipDate (if present)
# ============================================

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql.utils import AnalysisException
from datetime import date
import random

# --------------------------
# Configuration
# --------------------------
CATALOG = "workspace"
SCHEMA  = "adventureworks"

TABLE_INTERNET = "fact_internet_sales"
TABLE_RESELLER = "fact_reseller_sales"

def approx_1000():
    return 900 + int(random.random()*201)

N_INTERNET = approx_1000()
N_RESELLER = approx_1000()

D_START = date(2023, 1, 1)
D_END   = date(2025, 10, 3)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# --------------------------
# Column generators (logical values, before casting)
# --------------------------
def sales_amount_col():
    # [10..5000] with mean ≈ 1000
    return F.least(F.lit(5000.0),
                   F.greatest(F.lit(10.0),
                              F.lit(10.0) + (F.lit(5000.0)-F.lit(10.0)) * F.pow(F.rand(), F.lit(4.0))))

def freight_col():
    # U(10, 50)
    return F.lit(10.0) + F.rand()*F.lit(40.0)

def discount_col(sa):
    # 0 .. 10% of SalesAmount
    return (F.rand()*F.lit(0.10)) * sa

def tax_col(sa):
    # 18–22% of SalesAmount (≈20%)
    return (F.lit(0.18) + F.rand()*F.lit(0.04)) * sa

def order_date_col(d_start: date, d_end: date):
    span = (d_end - d_start).days
    return F.expr(f"date_add(to_date('{d_start.isoformat()}'), cast(floor(rand()*({span}+1)) as int))")

def due_and_ship_cols(od, d_end: date):
    d_end_lit = F.to_date(F.lit(d_end.isoformat()))
    due_delta = F.expr("cast(floor(rand()*42)+1 as int)")
    dd = F.least(F.date_add(od, due_delta), d_end_lit)  # date
    base_pick = F.expr("cast(floor(rand()*(DATEDIFF(dd, od)+1)) as int)")
    spill = F.when(F.rand() < F.lit(0.2),
                   F.expr("cast(floor(rand() * greatest(0, 42 - DATEDIFF(dd, od))) as int)")) \
             .otherwise(F.lit(0))
    ship_delta = F.least(F.lit(42), base_pick + spill)
    sd = F.least(F.date_add(od, ship_delta), d_end_lit)  # date
    return dd, sd

# --------------------------
# Type-aware casting helpers
# --------------------------
def cast_to_target(col_expr, target_type: T.DataType, for_date_time=False):
    """
    Cast a Column to the target Delta column type:
     - DATE  -> to_date
     - TIMESTAMP -> to_timestamp
     - DECIMAL(p,s) -> cast(DECIMAL(p,s))
     - DOUBLE/FLOAT -> cast(double/float)
     - INT/BIGINT/SMALLINT -> cast accordingly
     - else -> cast to target_type directly
    for_date_time: when True, col_expr is a DATE-like expression we may need to upcast to TIMESTAMP.
    """
    if isinstance(target_type, T.DateType):
        return F.to_date(col_expr)
    if isinstance(target_type, T.TimestampType):
        # upcast date -> timestamp at midnight
        if for_date_time:
            return F.to_timestamp(col_expr)
        return F.col(col_expr) if isinstance(col_expr, str) else F.to_timestamp(col_expr)
    if isinstance(target_type, T.DecimalType):
        return col_expr.cast(T.DecimalType(target_type.precision, target_type.scale))
    if isinstance(target_type, (T.DoubleType,)):
        return col_expr.cast("double")
    if isinstance(target_type, (T.FloatType,)):
        return col_expr.cast("float")
    if isinstance(target_type, (T.IntegerType,)):
        return col_expr.cast("int")
    if isinstance(target_type, (T.LongType,)):
        return col_expr.cast("bigint")
    if isinstance(target_type, (T.ShortType,)):
        return col_expr.cast("smallint")
    # default fallback
    return col_expr.cast(target_type.simpleString())

def overwrite_if_exists(df, target_schema, name_lc, new_col, for_date_time=False, round_money=False):
    """
    If column exists (case-insensitive), cast to target type and overwrite it.
    Optionally round money-like fields to 2 decimals after cast.
    """
    if name_lc in target_schema:
        tgt_name, tgt_type = target_schema[name_lc]
        col_casted = cast_to_target(new_col, tgt_type, for_date_time=for_date_time)
        if round_money and isinstance(tgt_type, (T.DecimalType, T.DoubleType, T.FloatType)):
            col_casted = F.round(col_casted, 2)
        return df.withColumn(tgt_name, col_casted)
    return df

# --------------------------
# Main inflator
# --------------------------
def inflate_table(table_name: str, n_rows: int):
    """
    Append ~n_rows records to `table_name` (no schema changes):
      - For each generated row, pick exactly one random donor using a partitioned window (no global window warning).
      - Overwrite only known measure/date columns (if present).
      - Output EXACTLY the table's columns in order, all cast to STRING (ACL-friendly, no schema merge).
    """
    from pyspark.sql import functions as F, Window
    from pyspark.sql.utils import AnalysisException

    fqn = table_name

    # Load donors and inspect exact table schema & column order
    try:
        donors = spark.table(fqn)
    except AnalysisException as e:
        raise RuntimeError(f"Table not found: {fqn}") from e

    if donors.limit(1).count() == 0:
        raise RuntimeError(f"Table {fqn} has no donor rows.")

    target_cols = donors.schema.fieldNames()  # exact order & casing
    has = {c.lower(): c for c in target_cols}  # case-insensitive presence check

    # ------------------------------------------------------------------
    # Generate n_rows rows and assign exactly one random donor per row
    # (Partitioned window by generated row id -> no global window warning)
    # ------------------------------------------------------------------
    gen = spark.range(n_rows).withColumnRenamed("id", "__seq")

    donors_randed = donors.withColumn("__rand", F.rand())
    candidates = gen.crossJoin(donors_randed)

    win = Window.partitionBy("__seq").orderBy(F.col("__rand"))
    picked = (candidates
              .withColumn("__rn", F.row_number().over(win))
              .filter(F.col("__rn") == 1)
              .drop("__rn", "__rand"))

    # ----- Build temporary logical columns (as DATE/DOUBLE), then format to STRING -----
    # Dates
    span_days = (D_END - D_START).days
    od_date = F.expr(f"date_add(to_date('{D_START.isoformat()}'), cast(floor(rand()*({span_days}+1)) as int))")
    d_end_lit = F.to_date(F.lit(D_END.isoformat()))
    dd_date = F.least(F.date_add(od_date, F.expr("cast(floor(rand()*42)+1 as int)")), d_end_lit)

    # Ship delta: usually between Order and Due; sometimes spills but ≤ 42 days total
    # Use temp columns to avoid unresolved references
    base = (picked
            .withColumn("__od_tmp", od_date.cast("date"))
            .withColumn("__dd_tmp", dd_date.cast("date")))
    base_pick = F.expr("cast(floor(rand()*(DATEDIFF(__dd_tmp, __od_tmp)+1)) as int)")
    spill = F.when(F.rand() < F.lit(0.2),
                   F.expr("cast(floor(rand() * greatest(0, 42 - DATEDIFF(__dd_tmp, __od_tmp))) as int)")) \
             .otherwise(F.lit(0))
    ship_delta = F.least(F.lit(42), base_pick + spill)
    df = base.withColumn("__sd_tmp", F.least(F.date_add(F.col("__od_tmp"), ship_delta), d_end_lit).cast("date"))

    # Measures (as DOUBLE, then formatted as string)
    sa = (F.lit(10.0) + (F.lit(5000.0) - F.lit(10.0)) * F.pow(F.rand(), F.lit(4.0)))
    sa = F.when(sa < 10.0, 10.0).when(sa > 5000.0, 5000.0).otherwise(sa)
    fr = F.lit(10.0) + F.rand()*F.lit(40.0)
    tx = (F.lit(0.18) + F.rand()*F.lit(0.04)) * sa
    ds = (F.rand()*F.lit(0.10)) * sa

    # Overwrite only if column exists; format to STRING to match table schema
    if "orderdate" in has:
        df = df.withColumn(has["orderdate"], F.date_format(F.col("__od_tmp"), "yyyy-MM-dd"))
    if "duedate" in has:
        df = df.withColumn(has["duedate"], F.date_format(F.col("__dd_tmp"), "yyyy-MM-dd"))
    if "shipdate" in has:
        df = df.withColumn(has["shipdate"], F.date_format(F.col("__sd_tmp"), "yyyy-MM-dd"))

    if "salesamount" in has:
        df = df.withColumn(has["salesamount"], F.format_number(sa, 2).cast("string"))
    if "taxamt" in has:
        df = df.withColumn(has["taxamt"], F.format_number(tx, 2).cast("string"))
    if "freight" in has:
        df = df.withColumn(has["freight"], F.format_number(fr, 2).cast("string"))
    if "discount" in has:
        df = df.withColumn(has["discount"], F.format_number(ds, 2).cast("string"))

    # ----- Project EXACT table schema (order & names), all as STRING -----
    # Drop helper cols if still present
    for extra in ["__seq", "__od_tmp", "__dd_tmp", "__sd_tmp"]:
        if extra in df.columns:
            df = df.drop(extra)

    # Ensure each target column exists and is string
    select_exprs = []
    for c in target_cols:
        if c not in df.columns:
            select_exprs.append(F.lit(None).cast("string").alias(c))
        else:
            select_exprs.append(F.col(c).cast("string").alias(c))

    df_out = df.select(*select_exprs)

    # Append WITHOUT schema change (ACL-friendly)
    df_out.write.mode("append").option("mergeSchema", "false").saveAsTable(fqn)

    return n_rows

# --------------------------
# Execute
# --------------------------
inserted_internet = inflate_table(TABLE_INTERNET, N_INTERNET)
inserted_reseller = inflate_table(TABLE_RESELLER, N_RESELLER)

# Verify totals
display(spark.sql(f"""
SELECT '{TABLE_INTERNET}' AS table_name, COUNT(*) AS total_rows FROM {TABLE_INTERNET}
UNION ALL
SELECT '{TABLE_RESELLER}' AS table_name, COUNT(*) AS total_rows FROM {TABLE_RESELLER}
"""))
